In [ ]:
import pandas as pd
import altair as alt
import scipy.stats as stats

alt.data_transformers.disable_max_rows()

In [ ]:
import matplotlib as mpl
mpl.rcParams['font.family'] = 'Arial'

@alt.theme.register('arial_theme', enable=True)
def arial_theme():
    return alt.theme.ThemeConfig({
        'config': {
            'font': 'Arial',
            'title': {'font': 'Arial'},
            'axis': {'labelFont': 'Arial', 'titleFont': 'Arial'},
            'legend': {'labelFont': 'Arial', 'titleFont': 'Arial'},
            'header': {'labelFont': 'Arial', 'titleFont': 'Arial'},
            'text': {'font': 'Arial'},
        }
    })

In [ ]:
original_data='./Data/final_tables/supplementary_file_1_BARD1_SGE_final_table.xlsx'
data_wd17='./Data/extra_data/20260723_BARD1.allscores_wD17.tsv'

In [ ]:
original_df = pd.read_excel(original_data, sheet_name='scores')
wd17_df=pd.read_csv(data_wd17, sep='\t')

wd17_df = wd17_df.loc[wd17_df['ref'].str.len()==1]
wd17_df['pos_id'] = wd17_df['pos'].astype(str) + ':' + wd17_df['alt']

# Inspect Scores for All Regions

## Initial Data Processing

In [ ]:
og_nod17 = original_df[['target', 'pos_id', 'functional_consequence', 'consequence','score']]
wd17_nod17=wd17_df[['target', 'pos_id', 'functional_consequence', 'score']]

wd17_nod17= wd17_nod17.rename(columns={'score': 'wd17_score'})

In [ ]:
merged_no_d17 = pd.merge(og_nod17, wd17_nod17, on=['target', 'pos_id'], how='inner')
merged_no_d17

## Scatter Plot

In [ ]:
non_d17_scatter = alt.Chart(merged_no_d17).mark_point().encode(
    x = 'score:Q',
    y='wd17_score:Q',
).facet('target:N', columns=5)

non_d17_scatter.display()

## Correlation Heatmap

In [ ]:
grouped = merged_no_d17.groupby('target')

output_tuples = []
for group, target_df in grouped:
    og_score=target_df['score']
    wd17_score=target_df['wd17_score']

    corr, _ = stats.pearsonr(og_score, wd17_score)

    return_tuple = (group, corr, 'orignal vs. w_d17')
    output_tuples.append(return_tuple)

corr_df = pd.DataFrame(output_tuples, columns=['target', 'correlation', 'test'])
print(corr_df)

In [ ]:
corr_heatmap = alt.Chart(corr_df).mark_rect().encode(
    x='test:N',
    y='target:N',
    color='correlation:Q',
    tooltip=['correlation']
)

corr_heatmap.display()

In [ ]:
print(merged_no_d17.value_counts('functional_consequence_x').reset_index())
print(merged_no_d17.value_counts('functional_consequence_y').reset_index())


# D17 Region Comparison

In [ ]:
d17s_only = merged_no_d17[merged_no_d17['target'].isin(['BARD1_X1A', 'BARD1_X1A;BARD1_X1B', 'BARD1_X7A', 'BARD1_X7A;BARD1_X7B', 'BARD1_X8A', 'BARD1_X8A;BARD1_X8B'])]

print(d17s_only)

In [ ]:
palette = [
    '#006616', # dark green,
    '#81B4C7', # dusty blue
    '#ffcd3a', # yellow
    '#6AA84F', # med green
    '#93C47D', # light green
    '#888888', # med gray
    '#000000', # black
    '#1170AA', # darker blue
    '#CFCFCF', # light gray
        
    ]
    
variant_types = [
    'synonymous_variant',
    'missense_variant',  
    'stop_gained',
    'intron_variant', 
    'UTR_variant',
    'stop_lost',
    'start_lost',
    'splice_site_variant', 
    'splicing_variant'
]

scatter_plot = alt.Chart(d17s_only).mark_point().encode(
    x=alt.X('score:Q', scale=alt.Scale(zero=False)),
    y=alt.Y('wd17_score:Q', scale=alt.Scale(zero=False)),
    color=alt.Color('consequence:N',
                    scale=alt.Scale(domain=variant_types,
                                    range=palette)
    ),
    tooltip=['pos_id', 'functional_consequence_x', 'functional_consequence_y']
).facet('target:N', columns = 2).resolve_scale(x='independent', y='independent')

scatter_plot.display()

In [ ]:
big_changes = d17s_only[((d17s_only['functional_consequence_x']=='functionally_normal') & (d17s_only['functional_consequence_y']=='functionally_abnormal')) |
                            ((d17s_only['functional_consequence_x']=='functionally_abnormal') & (d17s_only['functional_consequence_y']=='functionally_normal'))]

print(big_changes)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

d17s_only['target']=d17s_only['target'].str.split(';').str[0]
d17s_only['pct_d13'] = d17s_only.groupby('target')['score'].rank(pct=True) * 100
d17s_only['pct_d17'] = d17s_only.groupby('target')['wd17_score'].rank(pct=True) * 100
d17s_only['pct_shift'] = d17s_only['pct_d17'] - d17s_only['pct_d13']

targets = sorted(d17s_only['target'].unique())
palette = dict(zip(targets, plt.cm.tab10.colors))

fig, axes = plt.subplots(2, 1, figsize=(6.5, 9), gridspec_kw={'height_ratios': [1, 1.3]})

# --- Panel A: percentile shift by target ---
ax = axes[0]
rng = np.random.default_rng(0)
for i, t in enumerate(targets):
    sub = d17s_only.loc[d17s_only['target'] == t, 'pct_shift']
    x = i + 1 + rng.uniform(-0.15, 0.15, size=len(sub))
    ax.scatter(x, sub, s=10, alpha=0.5, color=palette[t], linewidths=0)
    med = sub.median()
    ax.plot([i + 0.75, i + 1.25], [med, med], color='black', lw=2, zorder=5)

ax.axhline(0, color='gray', ls='--', lw=1, zorder=0)
ax.set_xticks(range(1, len(targets) + 1))
ax.set_xticklabels(targets, rotation=30, ha='right')
ax.set_ylabel('Percentile shift (D17 − D13)')
ax.set_xlim(0.5, len(targets) + 0.5)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# --- Panel B: percentile-percentile scatter across targets ---
ax2 = axes[1]
for t in targets:
    sub = d17s_only[d17s_only['target'] == t]
    ax2.scatter(sub['pct_d13'], sub['pct_d17'], s=10, alpha=0.5,
                color=palette[t], label=t, linewidths=0)

ax2.plot([0, 100], [0, 100], color='gray', ls='--', lw=1, zorder=0)
ax2.set_xlabel('Percentile at D13')
ax2.set_ylabel('Percentile at D17')
ax2.set_xlim(0, 100)
ax2.set_ylim(0, 100)
ax2.set_aspect('equal')
ax2.legend(frameon=False, fontsize=8, loc='upper left', bbox_to_anchor=(1.02, 1))
for spine in ['top', 'right']:
    ax2.spines[spine].set_visible(False)

# Optional: highlight specific variants of interest by pos_id
# flagged_pos_ids = ['214745778:C', ...]
# flagged = df[df['pos_id'].isin(flagged_pos_ids)]
# ax2.scatter(flagged['pct_d13'], flagged['pct_d17'], s=60, marker='D',
#             facecolor='none', edgecolor='black', linewidths=1.5, zorder=6)

plt.tight_layout()
#plt.savefig('d13_d17_percentile_stability.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for t, sub in d17s_only.groupby('target'):
    r, _ = stats.pearsonr(sub['score'], sub['wd17_score'])
    rho, _ = stats.spearmanr(sub['score'], sub['wd17_score'])
    tau, _ = stats.kendalltau(sub['score'], sub['wd17_score'])
    print(f'{t} (n={len(sub)}): pearson r={r:.3f}  spearman rho={rho:.3f}  kendall tau={tau:.3f}')

In [ ]:
for t, sub in d17s_only.groupby('target'):
    sub = sub.copy()
    sub['tercile'] = pd.qcut(sub['score'], 3, labels=['bottom', 'middle', 'top'], duplicates='drop')
    print(t)
    for name, grp in sub.groupby('tercile', observed=True):
        tau, _ = stats.kendalltau(grp['score'], grp['wd17_score'])
        print(f'  {name}: tau={tau:.3f}, n={len(grp)}')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

palette = [
    '#006616',  # dark green
    '#81B4C7',  # dusty blue
    '#ffcd3a',  # yellow
    '#6AA84F',  # med green
    '#93C47D',  # light green
    '#888888',  # med gray
    '#000000',  # black
    '#1170AA',  # darker blue
    '#CFCFCF',  # light gray
]

variant_types = [
    'synonymous_variant',
    'missense_variant',
    'stop_gained',
    'intron_variant',
    'UTR_variant',
    'stop_lost',
    'start_lost',
    'splice_site_variant',
    'splicing_variant'
]

consequence_palette = dict(zip(variant_types, palette))
fallback_color = '#CFCFCF'

variants_of_interest = ['214809515:A', '214809539:A', '214809554:A', '214809491:A']

targets = sorted(d17s_only['target'].unique())

long_df = d17s_only.melt(
    id_vars=['target', 'pos_id', 'consequence'],
    value_vars=['score', 'wd17_score'],
    var_name='timepoint', value_name='sge_score'
)
long_df['timepoint'] = long_df['timepoint'].map({'score': 'D13', 'wd17_score': 'D17'})

is_voi = long_df['pos_id'].isin(variants_of_interest)
background_df = long_df[~is_voi]
voi_df = long_df[is_voi]

fig, axes = plt.subplots(1, len(targets), figsize=(4.5 * len(targets), 5), sharey=True)
if len(targets) == 1:
    axes = [axes]

for ax, t in zip(axes, targets):
    bg = background_df[background_df['target'] == t]
    voi = voi_df[voi_df['target'] == t]

    sns.violinplot(data=long_df[long_df['target'] == t], x='timepoint', y='sge_score', ax=ax,
                    inner=None, color='#f0f0ee', cut=0, linewidth=1, zorder=0)

    pal_bg = {c: consequence_palette.get(c, fallback_color) for c in bg['consequence'].unique()}
    sns.stripplot(data=bg, x='timepoint', y='sge_score', hue='consequence',
                  palette=pal_bg, ax=ax, alpha=0.6, size=3, jitter=0.25, legend=False)

    if len(voi):
        pal_voi = {c: consequence_palette.get(c, fallback_color) for c in voi['consequence'].unique()}
        sns.stripplot(data=voi, x='timepoint', y='sge_score', hue='consequence',
                      palette=pal_voi, ax=ax, size=8, linewidth=1, edgecolor='black',
                      jitter=0.2, legend=False, zorder=6)

    ax.set_title(t)
    ax.set_xlabel('')
    ax.axhline(0, color='gray', ls=':', lw=0.8, zorder=0)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

axes[0].set_ylabel('SGE score')

present_all = [c for c in variant_types if c in d17s_only['consequence'].unique()]
legend_elems = [Line2D([0], [0], marker='o', linestyle='', markersize=6,
                        color=consequence_palette[c], label=c) for c in present_all]
legend_elems.append(Line2D([0], [0], marker='o', linestyle='', color='white',
                             markeredgecolor='black', markersize=8, label='Variant of interest'))
fig.legend(handles=legend_elems, loc='center left', bbox_to_anchor=(1.01, 0.5),
           frameon=False, fontsize=8)

plt.tight_layout(rect=[0, 0, 0.85, 1])
#plt.savefig('d13_d17_by_target_consequence.pdf', dpi=300, bbox_inches='tight')
plt.show()